### <u> Generate a local "_Stars Appearing_" sequence: sonification + animation </u>

This builds the "_Stars Appearing_" piece from the "_Audible Universe_" planetarium
show for **any site and any night**, and renders a matching animation to go with it.

The sky is computed with `skyfield` from the _Hipparcos_ catalogue, sonified with
`strauss`, and animated as an **equirectangular** (360&deg; &times; 180&deg;)
panorama. A planetarium dome master comes from letting `ffmpeg`'s `v360` filter
reproject that panorama to fisheye &mdash; we never render fisheye ourselves.

The `.py` beside this notebook in `examples/multimedia/` holds the two extra non-`strauss` elements - the sky, from `skyfield`, and the animation, via `numpy`
and `ffmpeg`.

#### Requirements

Beyond `strauss` itself you need `skyfield`, and a working `ffmpeg` on your `PATH`.
The first run downloads the _Hipparcos_ catalogue (~50 MB) and the DE421 ephemeris
(~17 MB) into a cache directory; later runs reuse them.

The next cell does nothing at all unless it finds itself on _Google_ `Colab`, where
it fetches the repository and installs from it. It has to install `strauss` from the
repository rather than from `PyPI`, because this example depends on
`strauss.get_table()`, and it needs the clone regardless because the
instrument samples are not carried in the `PyPI` package.


In [ ]:
import sys

if "google.colab" in sys.modules:
    !git clone -b local_stars_appearing_animation --single-branch https://github.com/james-trayford/strauss.git
    !pip install --quiet ./strauss skyfield==1.53
    %cd strauss/examples/multimedia/


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import strauss

from StarsAppearingLocal import (Config, observed_sky, sky_panorama,
                                 facing_degrees, unit_scale, write_videos,
                                 CARDINALS)

### <u> Chosen properties </u>

Change these and re-run. The defaults describe the **Sherwood Observatory** site
looking south, at a relatively small size for quicker rendering. On _Colab_ these
appear as a form; everywhere else they are ordinary assignments.

In [ ]:
#@markdown ### Where and when
latitude = 53.1143737  #@param {type:"number"}
longitude = -1.2219389  #@param {type:"number"}
#@markdown Latitude is +ve north, longitude +ve *east* &mdash; 1.22&deg; west is `-1.22`.
date_time = "2026-03-25 18:45:00"  #@param {type:"string"}
time_zone = "Europe/London"  #@param {type:"string"}
#@markdown `facing` is the centre of the panorama, and of the stereo image.
facing = "S"  #@param ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"]
#@markdown Higher `mag_limit` includes more, dimmer stars.
mag_limit = 4  #@param {type:"slider", min:1, max:7, step:0.5}

#@markdown ### The sound
duration = 45  #@param {type:"number"}
system = "stereo"  #@param ["mono","stereo","5.1","7.1"]

#@markdown ### The picture
#@markdown `size` is `fast_preview` 512&times;256, `preview` 1024&times;512, or
#@markdown `full` &mdash; the star map's own 4096&times;2048.
size = "preview"  #@param ["fast_preview","preview","full"]
fps = 15  #@param {type:"integer"}
#@markdown `output` picks the equirectangular `panorama`, the fisheye `dome`
#@markdown master, or `both` from a single pass of the frame generator.
output = "both"  #@param ["panorama","dome","both"]
#@markdown `sky_exposure` is how bright the generated sky comes out, and
#@markdown `horizon` blacks out the ground beneath it.
sky_exposure = 0.35  #@param {type:"number"}
horizon = True  #@param {type:"boolean"}

outdir = "stars_appearing_preview"  #@param {type:"string"}

cfg = Config(latitude=latitude, longitude=longitude, date_time=date_time,
             time_zone=time_zone, facing=facing, mag_limit=mag_limit,
             duration=duration, system=system, size=size, fps=fps,
             output=output, sky_exposure=sky_exposure, horizon=horizon,
             outdir=outdir)

# `background` is not in the form: it defaults to "auto", which renders a
# panorama to match everything above (see the next section). Pass a path to
# `Config` to use an image of your own instead, or `None` for a black sky.
print(f"{cfg.width} x {cfg.height}, writing {cfg.output}")

### <u> The sky </u>

`skyfield` gives the altitude and azimuth of every catalogue star as seen from the
chosen site at the chosen instant. The catalogue is cut down by magnitude *before*
positions are computed, which is the difference between transforming ~118,000 stars
and ~1,500.

In [ ]:
sky = observed_sky(cfg)
print(f"{len(sky)} stars brighter than magnitude {cfg.mag_limit} above the horizon")
sky.head()

A quick look at what we are about to hear, as it would appear on the
panorama &mdash; the facing direction in the middle, the horizon along the bottom.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_facecolor("#0b0c15")

x = (sky["az"] - facing_degrees(cfg.facing) - 180) % 360

ax.scatter(x, sky["alt"], s=40 * 10 ** (-0.2 * sky["magnitude"]),
           c=sky["bv"], cmap="RdYlBu_r", vmin=-1.5, vmax=2.5, lw=0)
ax.set_xticks([(360 * i / 16 - facing_degrees(cfg.facing) - 180) % 360
               for i in range(16)])
ax.set_xticklabels(CARDINALS, fontsize=8)
ax.set_xlim(0, 360)
ax.set_ylim(0, 90)
ax.set_xlabel("compass direction")
ax.set_ylabel("altitude [deg]")
ax.set_title(f"{len(sky)} stars over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
             f"at {cfg.date_time}")
plt.show()

### <u> The sky to draw it on </u>

The pulses read far better over a real sky than over black, and the panorama to lay
them on is rendered from the same site, instant and facing as everything else &mdash;
so there is no image to supply, and nothing to keep in step by hand.

NASA's [*Deep Star Maps 2020*](https://svs.gsfc.nasa.gov/4851/) are all-sky
equirectangular images in **celestial** coordinates. Turning one into the view from
a particular place at a particular moment means asking, for every pixel of the
output, which point of the sky it looks at &mdash; which is `skyfield` again, run
backwards, from horizon coordinates to right ascension and declination.

That map is `OpenEXR`, and it is decoded here by `ffmpeg`, which this example
already needs. So this adds **no new dependencies** &mdash; only a 36&nbsp;MB
download on the first run, cached alongside the catalogue.

The named frame sizes are **2:1**, the shape of a 360&deg; &times; 180&deg;
panorama, and `"full"` is the star map's own 4096 &times; 2048 &mdash; so its pixels
are used as they are rather than resampled, and the sky is not stretched. A frame of
any other shape still works; the sky is simply stretched to fill it.

Raise `sky_exposure` to bring the Milky Way up; lower it to keep the sky dark and
let the star pulses carry the picture. `horizon` leaves everything below the
horizon black, as the ground would: no star is sounded from down there, and the
dome master needs it &mdash; otherwise the sky beneath your feet fills its corners
and there is no horizon left to see.

In [ ]:
background = sky_panorama(cfg)
print(background)

plt.figure(figsize=(14, 14 * cfg.height / cfg.width))
plt.imshow(plt.imread(background))
plt.axis("off")
plt.title(f"the sky over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
          f"at {cfg.date_time}, facing {cfg.facing}")
plt.show()

### <u> The sonification </u>

Uses`stars_appearing` `strauss` **style**. Four columns prepared, and the names must
match the `input:` names in the style - `sonify` matches a `DataFrame`'s
columns to the style by name:

- **`azimuth`** is the one that is easy to get backwards. `strauss` measures azimuth
  **anticlockwise from straight ahead**, while astronomical azimuth runs **clockwise
  from north**, so facing direction *minus* star azimuth is the right way round.
- **`polar`** is measured from the zenith down, not from the horizon up.
- **`volume`** quietens the dimmer stars. They are far more numerous, so without
  this the piece grows steadily louder as it goes.
- **`pitch_shift`** detunes each note very slightly, so that the many stars sharing
  a note do not phase against one another.

In [ ]:
smag = unit_scale(sky["magnitude"].to_numpy(float))
rng = np.random.default_rng(cfg.seed + 1)

frame = pd.DataFrame({
    "magnitude": sky["magnitude"].to_numpy(float),
    "colour":    sky["bv"].to_numpy(float),
    "azimuth":   (facing_degrees(cfg.facing) - sky["az"].to_numpy(float)) % 360,
    "polar":     90.0 - sky["alt"].to_numpy(float),
    "volume":    (1 - smag) ** 0.5,
    "pitch_shift": 5e-3 * rng.random(len(sky)),
}, index=[f"HIP{hip}" for hip in sky.index])

frame.head()

Now sonify it. `source_names` labels each event with its star, so the table below
can be joined back to the catalogue, and `angle_unit="degrees"` is what makes that
table report real degrees rather than raw 0-1 fractions &mdash; without it the
animation would be driven by nonsense.

In [ ]:
strauss.sonify(frame, style="stars_appearing", channels=cfg.system,
               duration=cfg.duration, angle_unit="degrees",
               source_names=list(frame.index))

strauss.display()

### <u> What sounded, and when </u>

This is the join between sound and picture. `get_table()` reports what is actually
heard &mdash; the time of each note in seconds, the note itself, and the star's
angles in degrees &mdash; and the animation reads its timings from here rather than
working them out again.

The table carries its units in a second column level, which is for reading rather
than for arithmetic, so the animation takes the plain names and picks up the
magnitude and colour it needs for pulse size and colour from the frame we sonified.

In [ ]:
table = strauss.get_table()
display(table.head(10))

# the units sit in a second column level, which is for reading rather than for
# arithmetic - drop to the plain names before touching the numbers
flat = table.copy()
flat.columns = flat.columns.get_level_values(0)

events = pd.DataFrame({
    "time":    flat["Time"].to_numpy(float),
    "azimuth": flat["Azimuthal Angle"].to_numpy(float),
    "polar":   flat["Polar Angle"].to_numpy(float),
}, index=flat["Source"].to_numpy())

events = events.join(frame[["magnitude", "colour"]]).sort_values("time")
events.head()

### <u> The animation </u>

Each star is a pulse that swells and fades as its note sounds. Frames are generated
as raw `RGBA` and piped straight into `ffmpeg`, which lays them over the background,
muxes the audio, and writes the finished video &mdash; nothing touches the disk in
between.

A star is only drawn while it is bigger than half a pixel, which for these
envelopes is around a second, so only the handful of stars actually alive in each
frame get any work done on them.

`strauss.save()` writes the audio with no caption embedded &mdash; a caption is
*prepended* to the track, which would slide the whole thing against the picture.

`output` above chooses what comes out. Asked for `both`, they come from a
single pass of the frame generator rather than drawing everything twice:

- the **panorama**, equirectangular, 360&deg; across by 180&deg; high
- the **dome master**, the same pixels reprojected to a 180&deg; fisheye by
  `ffmpeg`'s `v360` filter and tilted so the zenith lands in the centre of the
  dome, the horizon falling on the rim of the inscribed circle and the facing
  direction at the bottom, which is the usual dome-master orientation.

Whichever you asked for is written to disk. Because the panorama underneath was rendered from the
same site and instant as the sonification, each flash lands on the star it is
sounding for, picking out the constellations as the piece goes on.

In [ ]:
audio = cfg.outdir / "stars_appearing.wav"
cfg.outdir.mkdir(parents=True, exist_ok=True)
strauss.save(str(audio))

# with no targets given, `cfg.output` decides which videos get written
videos = write_videos(cfg, events, audio, sky=background)

In [ ]:
for path in (audio, *videos):
    print(f"{path.stat().st_size / 1e6:8.1f} MB  {path}")

In [ ]:
# on Colab use moviepy to display the finished videos inline
if "google.colab" in sys.modules:
    import moviepy.editor
    for path in videos:
        display(moviepy.editor.ipython_display(f"{path}"))

### <u> Tidying up, and going bigger </u>

`strauss.close()` finishes with this figure, so that a re-run starts clean rather
than adding a second sonification alongside the first:

```python
strauss.close()
```

Rendering time is dominated by frame size. The preview settings here take a few
seconds. A full 4096 &times; 2160, 60-second sequence at 30 fps in `5.1`, giving both
the panorama and the dome master, can take a while to run (hardware and environment
dependent).

Render at preview size while you are still choosing a site and a night, then raise
`width`, `height` and `fps` for the final pass. If you want a different *sound*,
that is now a change of style rather than a change of code &mdash; `strauss.sonify`
takes any style name, and `stars_appearing` is only the one this piece was built
around.

In [ ]:
strauss.close()